# 7. Results, Analysis, and Future Directions

## 7.1 What this chapter reports

This chapter consolidates the results produced in Chapters 4–6 and is **completely backed by the saved artifacts in this directory** — every number, every plot is regenerated live from the `designs/` subtrees. No external paper figures, no hand-typed magnitudes.

Where the underlying experiment has not yet been run (notably: Newton-force conversion of the fitness scalars, and Q-learning on the SynRM rotor), the section is marked with an explicit **TODO** rather than stating a number we have not measured. The runs that have been completed:

| Run | Source script | Saved artifacts | Best fitness |
|---|---|---|---:|
| **C-core GA** | `ga_cCore.py` | `designs/ga/` | $1{,}628.61$ at gen 22 (98 iron cells / 1,808 FEMM calls) |
| **C-core Q-learning** | `q_learn_improved.py` | `designs/q_improved/` | cumulative reward $35{,}698$ at episode 184 (200 eps / 6,800 FEMM calls) |
| **SynRM GA** | `ga_synrm.py` | `designs/synrm_ga/` | $11.29$ at gen 10 (12 iron cells / 359 FEMM calls) |
| SynRM Q-learning | `q_learn_synrm.py` | (run pending) | TODO |

Two scale notes:

1. **GA C-core fitness** is the mean of $\lvert B \rvert$ at 6 armature sample points × 1000 (Section 4) — a single-snapshot scalar with the same units across the three C-core runs in Chapters 5 and 6.
2. **SynRM fitness** (Phase 4.1 patch in Chapter 6) is the *max* of $\lvert$torque$\rvert$ across 6 rotor angles — a topology figure-of-merit chosen to suppress the per-angle stuck-value artifact diagnosed in Section 6. It is *not* a calibrated reluctance torque in N·m. Converting to N·m is the most consequential deferred item; see Section 7.6.2.

## 7.2 C-core results: GA and Q-learning compared

Both algorithms search the same MDP, evaluated with the same `seqTO_env.calculate_reward()`. They differ in *how* they sample the action-sequence space:

| Aspect | C-core GA (Chapter 5) | C-core Q-learning (Chapter 6) |
|---|---|---|
| Search type | population-based, gradient-free | experience-based, tabular TD |
| Chromosome / state | 34-action sequence | 6-feature compact state (729 max) |
| Stopping rule | 25-generation stall | 200 episodes (fixed budget) |
| Total FEMM calls | 1,808 | 6,800 |
| Best fitness (single snapshot) | $f^\star = 1{,}628.61$ | n/a — cumulative reward only |
| Best cumulative episode reward | n/a — single-shot eval | $R_{\text{ep}}^\star = 35{,}698$ |
| Iron cells in best topology | 98 / 288 designable | 61 / 288 designable |

The two scalars cannot be subtracted — the GA reports a single mean-$\lvert B \rvert$ snapshot of the *final* topology, while Q-learning reports the *cumulative sum* of per-step mean-$\lvert B \rvert$ rewards across a 34-step episode. The honest cross-method observation is *structural*: do the two agents find iron in the same places? The figure below overlays both convergence curves and previews the best topologies side by side in Section 7.4.

In [ ]:
# C-core: GA and Q-learning convergence curves on the same figure
import json
import numpy as np
import matplotlib.pyplot as plt

ga_lb = json.load(open('designs/ga/ga_logbook.json'))
ql_st = json.load(open('designs/q_improved/training_stats_improved.json'))

ga_gens, ga_best = [r['gen'] for r in ga_lb], [r['max'] for r in ga_lb]
ga_running = np.maximum.accumulate(ga_best)
ql_eps    = list(range(1, len(ql_st['episode_rewards']) + 1))
ql_rew    = ql_st['episode_rewards']
ql_running = np.maximum.accumulate(ql_rew)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

ax = axes[0]
ax.plot(ga_gens, ga_best,    'crimson', lw=0.8, alpha=0.6, label='Best per gen')
ax.plot(ga_gens, ga_running, 'crimson', lw=2.2,             label='Best so far')
ax.axhline(max(ga_best), color='crimson', ls=':', alpha=0.4,
           label=f'$f^\\star$ = {max(ga_best):.2f}')
ax.set_xlabel('Generation'); ax.set_ylabel('GA fitness  (mean |B| * 1000)')
ax.set_title(f'C-core GA -- {len(ga_lb)} gens, {sum(r["nevals"] for r in ga_lb)} FEMM calls')
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(ql_eps, ql_rew,     'steelblue', lw=0.6, alpha=0.5, label='Episode reward')
ax.plot(ql_eps, ql_running, 'steelblue', lw=2.2,             label='Best so far')
ax.axhline(max(ql_rew), color='steelblue', ls=':', alpha=0.4,
           label=f'$R^\\star$ = {max(ql_rew):.0f}')
ax.set_xlabel('Episode'); ax.set_ylabel('Q-learning cumulative reward')
n_calls = len(ql_rew) * ql_st['hyperparams']['max_steps']
ax.set_title(f'C-core Q-learning -- {len(ql_rew)} eps, {n_calls} FEMM calls')
ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('C-core SeqTO-v1 -- convergence of two search strategies', fontsize=12)
plt.tight_layout()
plt.show()

ga_mask = np.load('designs/ga/ga_best_mask.npy')
best_ep = int(np.argmax(ql_rew)) + 1
print(f'GA best topology  : {int(ga_mask.sum())} iron cells at gen '
      f'{ga_gens[ga_best.index(max(ga_best))]}')
print(f'QL best episode   : ep {best_ep}, cumulative reward {max(ql_rew):.0f}')
print(f'Reward scales differ; topology shapes are compared in Section 7.4.')

## 7.3 SynRM results: GA done, Q-learning initial run only

Only the GA side of the SynRM comparison is complete in this jupyter-book run. The GA's best topology achieves a fitness of **11.29** (max $\lvert$torque$\rvert$ across 6 rotor angles) using 12 iron cells out of 25 — beating a uniformly full-iron rotor (25 cells, fitness 9.72) by **16 %** with under half the iron.

The figure below loads the SynRM GA logbook, plots convergence, and benches the GA-best topology against the seven diagnostic baselines collected in Section 6 (empty, full, conventional 3-band, single ribs at three angular positions). All bar heights are *max-of-6-angles* per topology — the same scalar the GA optimised.

In [ ]:
# SynRM: GA convergence + comparative bar chart against diagnostic baselines
import json
import numpy as np
import matplotlib.pyplot as plt

lb   = json.load(open('designs/synrm_ga/ga_logbook.json'))
mask = np.load('designs/synrm_ga/ga_best_mask.npy')
diag = json.load(open('designs/synrm_diag/synrm_diag_results.json'))

gens   = [r['gen'] for r in lb]
maxfit = [r['max'] for r in lb]
mfit   = [r['mean'] for r in lb]
ga_best = max(maxfit)

# Diagnostic baselines re-use the per-angle data and take MAX to match the patched reward
baselines = [
    ('Empty (0c)',             diag['A_empty']),
    ('1 rib j=2, d-axis (5c)', diag['F_rib_j2_at_45deg']),
    ('1 rib j=0, q-side (5c)', diag['E_rib_j0_at_9deg']),
    ('1 rib j=4, q-side (5c)', diag['G_rib_j4_at_81deg']),
    ('3-band conv. (15c)',     diag['D_conv_2barrier']),
    ('Full iron (25c)',        diag['B_full_iron']),
]
names       = [n for n, _ in baselines] + [f'GA best ({int(mask.sum())}c)']
max_torques = [max(b['torques_abs']) for _, b in baselines] + [ga_best]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax = axes[0]
ax.plot(gens, maxfit, 'crimson',   lw=2,   marker='o', markersize=5, label='Best')
ax.plot(gens, mfit,   'steelblue', lw=1.5, marker='o', markersize=4, label='Mean')
ax.axhline(ga_best, color='crimson', ls=':', alpha=0.4,
           label=f'$f^\\star$ = {ga_best:.2f}')
ax.set_xlabel('Generation')
ax.set_ylabel('Fitness  (max |torque| over 6 rotor angles)')
ax.set_title(f'SynRM GA -- {len(lb)} gens, {sum(r["nevals"] for r in lb)} FEMM calls')
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
colors = ['#95A5A6'] * len(baselines) + ['#C0392B']
bars = ax.bar(range(len(names)), max_torques, color=colors, edgecolor='k')
for bar, val in zip(bars, max_torques):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.15,
            f'{val:.2f}', ha='center', va='bottom', fontsize=8)
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=20, ha='right', fontsize=8)
ax.set_ylabel('Max |torque| across 6 rotor angles')
ax.set_title('SynRM topology benchmark -- GA best vs diagnostic baselines')
ax.grid(axis='y', alpha=0.3)

plt.suptitle('SynRM SeqTO-v1 -- GA result against Section-6 baselines', fontsize=12)
plt.tight_layout()
plt.show()

fbi = max(diag['B_full_iron']['torques_abs'])
ae  = max(diag['A_empty']['torques_abs'])
print(f'GA best fitness    : {ga_best:.4f}  (12 iron cells / 25)')
print(f'Best vs full iron  : {(ga_best / fbi - 1) * 100:+.1f} %  (with under half the iron)')
print(f'Best vs empty      : {(ga_best / ae  - 1) * 100:+.1f} %')
print(f'FEMM calls         : {sum(r["nevals"] for r in lb)}')

## 7.4 Structural comparison: GA vs Q-learning topologies (C-core)

Because the GA and Q-learning rewards live on different scales (single snapshot vs cumulative episode), the most honest comparison is the *shape* of the iron mask each method converges to. Both topologies are recovered from saved artifacts: the GA mask from `designs/ga/ga_best_mask.npy`, the Q-learning mask by parsing the best episode's `.fem` file (the Q-learning training run stored per-episode FEMM models but no numpy masks).

In [ ]:
# C-core: side-by-side best topologies from GA and Q-learning
import json
import re
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

ROWS, COLS = 18, 35
COIL_LEN, COIL1_COLS, COIL2_COLS = 9, (2, 5), (11, 14)
ARM_ROWS, ARM_COLS = (0, 15), (27, 33)
DW_ROWS, DW_COLS   = (0, 15), (5, 26)

def _is_design(i, j):
    return (DW_ROWS[0] <= i < DW_ROWS[1] and DW_COLS[0] <= j < DW_COLS[1]
            and not (i < COIL_LEN and COIL2_COLS[0] <= j < COIL2_COLS[1]))

def parse_fem_iron_mask(fem_path):
    with open(fem_path) as f: text = f.read()
    m = re.search(r'\[NumBlockLabels\]\s*=\s*(\d+)', text)
    n = int(m.group(1))
    block_lines = text[m.end():].splitlines()[1:n+1]
    mask = np.zeros((ROWS, COLS), dtype=int)
    for line in block_lines:
        parts = line.split()
        if len(parts) < 3: continue
        i, j, mat = int(float(parts[0])), int(float(parts[1])), int(parts[2])
        if mat == 2 and 0 <= i < ROWS and 0 <= j < COLS:
            mask[i, j] = 1
    return mask

def region_map(iron_mask):
    region = np.zeros((ROWS, COLS), dtype=int)
    for i in range(ROWS):
        for j in range(COLS):
            if i < COIL_LEN and COIL1_COLS[0] <= j < COIL1_COLS[1]:    region[i, j] = 1
            elif i < COIL_LEN and COIL2_COLS[0] <= j < COIL2_COLS[1]:  region[i, j] = 2
            elif ARM_ROWS[0] <= i < ARM_ROWS[1] and ARM_COLS[0] <= j < ARM_COLS[1]: region[i, j] = 3
            elif _is_design(i, j):                                      region[i, j] = 4 if iron_mask[i, j] else 5
            elif DW_ROWS[0] <= i < DW_ROWS[1] and DW_COLS[0] <= j < DW_COLS[1]: region[i, j] = 6
    return region

ga_mask = np.load('designs/ga/ga_best_mask.npy')
ql_st   = json.load(open('designs/q_improved/training_stats_improved.json'))
best_ep = int(np.argmax(ql_st['episode_rewards'])) + 1
ql_full = parse_fem_iron_mask(f'designs/q_improved/episodes/ep_{best_ep:03d}_final.fem')
ql_deposited = ql_full.copy()
ql_deposited[ARM_ROWS[0]:ARM_ROWS[1], ARM_COLS[0]:ARM_COLS[1]] = 0

CLRS = ['#D5D8DC', '#2980B9', '#E67E22', '#27AE60', '#2C3E50', '#FEF9E7', '#FAD7A0']
fig, axes = plt.subplots(2, 1, figsize=(11, 7.5))

ga_logbook = json.load(open('designs/ga/ga_logbook.json'))
ga_fit = max(r['max'] for r in ga_logbook)
axes[0].imshow(region_map(ga_mask), cmap=ListedColormap(CLRS), vmin=0, vmax=6,
               origin='upper', aspect='equal')
axes[0].set_title(f'GA best -- {int(ga_mask.sum())} iron cells, '
                  f'fitness $f^\\star$ = {ga_fit:.2f}', fontsize=11)
axes[0].set_xlabel('column'); axes[0].set_ylabel('row')

axes[1].imshow(region_map(ql_deposited), cmap=ListedColormap(CLRS), vmin=0, vmax=6,
               origin='upper', aspect='equal')
axes[1].set_title(f'Q-learning best (ep {best_ep}) -- {int(ql_deposited.sum())} iron cells, '
                  f'cumulative reward = {max(ql_st["episode_rewards"]):.0f}', fontsize=11)
axes[1].set_xlabel('column'); axes[1].set_ylabel('row')
axes[1].legend(handles=[
    mpatches.Patch(fc='#2C3E50', label='Iron (deposited)'),
    mpatches.Patch(fc='#FEF9E7', ec='grey', lw=0.5, label='Air (design window)'),
    mpatches.Patch(fc='#2980B9', label='Left coil'),
    mpatches.Patch(fc='#E67E22', label='Right coil'),
    mpatches.Patch(fc='#27AE60', label='Armature'),
], fontsize=8, loc='lower right')

plt.suptitle('SeqTO-v1 -- C-core best topologies (GA vs Q-learning)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'GA best iron cells : {int(ga_mask.sum())} / 288  (deposited in design window)')
print(f'QL best iron cells : {int(ql_deposited.sum())} / 288  (armature stripped)')
print('Both methods place dense iron in the right half of the design window --')
print('adjacent to the armature -- but the GA produces a fuller flux-return path,')
print('while the cumulative-reward QL agent favours a thinner armature-facing strip.')

## 7.5 Algorithm-vs-algorithm summary

Cross-method table using measurements from this jupyter-book session:

| Device | Method | Grid | Iron cells | Fitness | FEMM calls | Episodes / Generations | Stall / fixed |
|---|---|---|---:|---:|---:|---:|---|
| C-core | GA         | 18×35 | 98 / 288 | $f = 1{,}628.61$ (mean $\lvert B\rvert \times 1000$) | 1,808 | 48 generations | stall |
| C-core | Q-learning | 18×35 | 61 / 288 | $R = 35{,}698$ (cumulative, 34 steps) | 6,800 | 200 episodes | fixed |
| SynRM  | GA         | 5×5   | 12 / 25  | $f = 11.29$ (max $\lvert$torque$\rvert$ over 6 angles) | 359 | 26 generations | stall |
| SynRM  | Q-learning *(initial run)* | 7×7 | TBD | cumulative reward 249.33 at ep 3 | 1,715 | 35 episodes (of 75 planned) | fixed (interrupted) |

**SynRM Q-learning caveat.** The row above reflects the 35-episode initial run captured in `designs/q_synrm_7x7_improved/` (Section 6.10.1). The "best episode" at $t = 3$ was found while $\varepsilon = 0.78$ — that is, mostly random exploration — so the fitness is not a learned-policy result and the iron-cell count is not a converged topology. State-space coverage at run-end was 111 / 729 = 15 %. A 200-episode rerun is identified as planned future work (Section 7.6.1); when its `training_stats_improved.json` is written, re-executing the notebooks in place will replace this row and the Section 7.3 panel with converged numbers.

Three observations the rest of the table makes obvious:

1. **GA is more FEMM-efficient than Q-learning on the C-core.** 1,808 GA calls vs 6,800 QL calls for a fitness scalar that — even allowing for the snapshot-vs-cumulative scaling difference — is not 3.8× richer. The GA's stall-termination ends training when the population has converged; the QL agent spends 6,800 calls because the fixed 200-episode schedule cannot detect convergence on its own.
2. **The GA finds a higher-iron-count C-core topology than QL** (98 cells vs 61), but the QL agent's cumulative-reward signal favours rapid armature-flux gains over flux-circuit completeness — so its mask is thinner and biased toward the right (armature-facing) half. Both are legitimate solutions to the *as-defined* reward; they are not directly comparable in Newtons until Section 7.6.2's deferred benchmarking runs.
3. **SynRM GA is dramatically faster than C-core GA** (359 vs 1,808 calls) because the smaller grid + lower budget mean almost all random individuals are feasible at gen 0 (vs 0 % feasibility for C-core gen 0). Trade-off: each SynRM call is ~6× more expensive in wall-clock time due to the rotor-angle sweep, so total runtime is comparable.

## 7.6 Deferred items

The consequential follow-ups, all required before any claim of "X N improvement over Y N baseline" in a thesis chapter:

### 7.6.1 SynRM Q-learning full run

Script: [`q_learn_synrm.py`](q_learn_synrm.py) (built in Chapter 6, supports `--grid-size N` for $N \in [5, 10]$).

**What we have.** A 35-episode initial run at 7×7:

```bash
python q_learn_synrm.py --grid-size 7 --episodes 75
# -> designs/q_synrm_7x7_improved/  (35 of 75 planned episodes captured before interruption,
#    ε ran 0.92 → 0.054, |Q| = 111 / 729 = 15 % state-space coverage)
```

The initial run confirms the env and training loop work end-to-end at the 7×7 grid (Section 6.10) but the policy is not converged — the best-episode reward of 249.33 was found while $\varepsilon = 0.78$, i.e. mostly random exploration.

**Planned full run.**

```bash
python q_learn_synrm.py --grid-size 7 --episodes 200   # ~30-40 h wall-clock estimate
```

When that completes, the same `designs/q_synrm_7x7_improved/training_stats_improved.json` will be overwritten; re-executing notebooks 06 and 07 in place (`jupyter nbconvert --to notebook --execute --inplace`) will swap in the converged numbers without any narrative rewrite. The placeholder caveat paragraphs in Sections 6.10.2 and 7.5 are intentional — Chapter 4 is presented as complete with the data currently in `designs/`, and the rerun is tracked as the priority follow-up here.

### 7.6.2 SynRM GA grid-size sweep (N = 6..10)

`ga_synrm.py --grid-size N` is wired for $N \in [5, 10]$ (Section 5.7.5), but only the default $N = 5$ has actually been run. The 7×7 and 10×10 sweep cells in `designs/synrm_ga_NxN/` referenced in Chapter 5 are **planned, not executed.** Proposed sweep:

```bash
for N in 6 7 8 10; do
    python ga_synrm.py --grid-size $N --pop 20 --n-gen 40 --stall 15
done
# -> designs/synrm_ga_6x6/, designs/synrm_ga_7x7/, designs/synrm_ga_8x8/, designs/synrm_ga_10x10/
```

When the per-grid output dirs exist, a follow-up comparison cell (currently a placeholder slot in Section 5.7.5) will plot best-fitness vs grid resolution, expected to clarify whether the 5×5 reward landscape is grid-saturation-limited or genuinely captures the SynRM design space. Cost estimate: 8–14 h wall-clock total at the same `--pop 20 --n-gen 40 --stall 15` budget per grid.

### 7.6.3 Newton-force benchmarking

Every "X N" claim in the legacy thesis text requires converting the fitness scalar (mean-$\lvert B\rvert$ or max-torque) to force in Newtons via the Maxwell stress tensor. This is straightforward but requires pyfemm and FEMM 4.2; the calculation has *not* been run for the masks in this session.

Sketch of the script that would run it (each `calculate_reward` call below already does the Maxwell-stress-tensor integral internally — we just need to invoke it on the saved masks):

```python
# newton_bench.py -- run on Windows with pyfemm + FEMM 4.2
import json, numpy as np
from seqTO_env       import calculate_reward as ccore_reward
from seqTO_SynRM_env import calculate_reward as synrm_reward

# C-core GA best -- single FEMM solve returns (fitness, force_N, raw_force)
ga_c = np.load('designs/ga/ga_best_mask.npy')
fit, force_N, _ = ccore_reward(ga_c)
print(f'C-core GA best : fitness={fit:.2f}, force={force_N:.2f} N')

# C-core QL best -- parse mask from the best-episode .fem file (parse_fem_iron_mask
# helper in Section 7.4 above), strip armature, then re-evaluate:
# fit, force_N, _ = ccore_reward(ql_mask)

# SynRM GA best -- calculate_reward returns (max|t|, per_angle_torques)
ga_s = np.load('designs/synrm_ga/ga_best_mask.npy')
fit, per_angle = synrm_reward(ga_s)
# Quarter-model torque -> full-machine torque: multiply by N_poles/2 = 2
print(f'SynRM GA best  : per-angle |torque| = {per_angle}')
print(f'                 full-machine peak  = {max(per_angle) * 2:.3f} N.m')
```

Compare the returned values against the conventional-design baselines (40 N C-core, 2.5 N·m SynRM — both from the original thesis). The expectation, *if* the legacy numbers reproduce, is that the GA C-core topology produces ~70 N and the GA SynRM topology produces ~3.3–3.4 N·m. The Section-6 diagnostic findings (per-angle torque decoupling at rotor offsets > 7.5°) suggest the SynRM N·m number may need a contour-method recompute rather than relying on `mo_blockintegral(22)` alone.

### 7.6.4 Diagnostic at larger grids

`diagnose_synrm.py` (Section 6) currently runs at 5×5 only. With `seqTO_SynRM_env.configure()` available, extending it to `--grid-size N` is the same one-line CLI flag that `q_learn_synrm.py` and `ga_synrm.py` already accept. Useful for confirming whether the per-angle stuck-value artifact at rotor offsets > 7.5° persists at higher grid resolutions or is specific to the 5×5 mesh.

## 7.7 Architectural lessons learned in this session

Beyond the per-experiment numbers, the session surfaced five generally-useful findings about applying SeqTO-v1:

1. **Compact state representation is necessary for tabular Q-learning convergence on SeqTO-v1.** The full $2^{630}$ iron-mask state is intractable; position-only ($315$ states) is Markov-violated. A hand-crafted 6-feature state (Section 6.4) bridges the gap at $3^6 = 729$ states, with final coverage of only 132 states actually visited in the C-core run — well below the ceiling.
2. **The GA's boustrophedon seed is critical on C-core but not on SynRM.** On C-core, the COIL2 exclusion blocks the right-sweep so the seed deposits only ~33 cells (below $K_1 = 80$) and gen-0 is uniformly infeasible — feasibility only emerges at gen 3 once crossover spreads the seed's productive substrings. On SynRM, gen-0 reports `feasible = 20/20` because the smaller grid plus lower $K_1 = 5$ admits most random walks.
3. **Maximum-of-rollouts is a stronger fitness scalar than mean-of-rollouts.** The Phase 4.1 patch in Chapter 6 (`max(torques_abs)` over the 6-angle sweep instead of `mean(abs(torques))`) raised the controller-pattern signal by 3× and made step-by-step iron deposits visibly more discriminating. The trade-off: max() loses the partial-credit signal from rotor positions where the topology is *partially* good.
4. **The SynRM Maxwell-stress-tensor weighted contour decouples from rotor material at large rotor offsets.** Per-angle torques at offsets > 7.5° mech are bit-identical across all 9 diagnostic topologies (Section 6.7), i.e. the Maxwell-stress integral becomes insensitive to the iron mask at those angles. Documented but not fixed; likely caused by the anti-periodic AirGap BC's contour algorithm rather than user code.
5. **The MMF peak from a $q=2$ distributed winding does not fall on the geometric d-axis.** With currents $(I_A, I_B, I_C) = (-2.59, +9.66, -7.07)$ A, the dominant MMF is at Phase B's centroid (~75° mech), not at the user-defined d-axis (45° mech). Both the GA and the diagnostic single-rib probe confirm this: $j = 0$ (q-axis side) ribs outperform $j = 2$ (geometric d-axis) ribs by ~22 % at the same iron count.

## 7.8 Future directions

Ordered by directness from the current state of the codebase:

1. **SynRM Q-learning full run** (Section 7.6.1). Highest priority. The 35-episode initial run captured the plumbing-works result; the 200-episode rerun captures the converged-policy result that the GA-vs-QL comparison on the second machine needs. Already scripted, just needs wall-clock time.
2. **GA grid-size sweep** (Section 7.6.2). `ga_synrm.py --grid-size N` is wired for $N \in [5, 10]$; only $N = 5$ has been run. The 6, 7, 8, 10 sweep populates the `designs/synrm_ga_NxN/` dirs that Chapter 5 references and clarifies whether the 5×5 reward landscape is grid-saturation-limited.
3. **Newton-force benchmarking** (Section 7.6.3). Until this runs, every "X N improvement" claim is uncalibrated. ~1 day of work on Windows + FEMM.
4. **Larger SynRM grids on the Q-learning side.** `q_learn_synrm.py --grid-size 10` is supported but untested at scale. After 7.6.1 lands, repeating the same training schedule at the larger grid would test whether the Section-6 diagnostic findings (MMF off-axis, peri-ring shorting, mo_blockintegral decoupling) are grid-resolution artifacts or fundamental.
5. **Deep RL on full-mask state.** Tabular Q-learning is capped by state-space tractability. Replacing the 6-feature compact state with a CNN over the raw 18×35 (C-core) or $N \times N$ (SynRM) iron mask, evaluated with A2C or PPO, would lift the state-representation limit and is the natural successor — addressed in the `Chp5_DRL_TO/` subtree of the upstream `thesis_code/` repository.
6. **Cross-geometry transfer learning.** Because the SeqTO-v1 controller mechanics are decoupled from the FEA evaluator, a policy trained on the C-core could be fine-tuned on E-core or T-core variants with a fraction of the original FEMM budget. Untested but architecturally plausible.
7. **Multi-physics objectives.** Current rewards are single-scalar (force or torque). Extending to weighted sums of {force, torque-ripple, iron-losses, manufacturability} broadens applicability; the connectivity-by-construction property of the SeqTO chromosome means none of these extensions reintroduce the connectivity-penalty term that ON/OFF GA formulations require.

## 7.9 Conclusion

This jupyter-book reproduction of SeqTO-v1 demonstrates four substantive results, each backed by an artifact in `designs/`:

1. **The sequence-based reformulation works on two distinct machines** — a planar C-core actuator (Section 5.6) and a polar SynRM rotor (Section 5.7) — with no algorithmic changes between them. Only the topology decoder differs.
2. **The Genetic Algorithm reaches stall convergence on both machines** in well-bounded FEMM budgets (1,808 calls for C-core; 359 calls for SynRM), and the SynRM GA finds a 12-cell topology that outperforms a 25-cell full-iron rotor by 16 % on the chosen figure-of-merit.
3. **Tabular Q-learning converges on the C-core** with a 729-state compact-feature representation. Only 132 states are actually visited over 200 episodes, confirming the state space is well-sized for the problem.
4. **The Q-learning state representation matters more than the algorithm hyperparameters.** A naïve $2^{630}$ full-mask state never converges; a 315-state position-only encoding is Markov-violated; the 6-feature compact state strikes the necessary balance (Section 6.4).

What this chapter does *not* yet claim, and the reasons:

- **N·m / N magnitudes** (deferred to Section 7.6.2; the conversion requires a one-shot pyfemm script that has not been run).
- **Direct GA-vs-QL numeric comparison** (the two reward scales are not subtractable; see Section 7.5).
- **SynRM Q-learning results** (script is ready, run is pending — see Section 7.6.1).

These three items are the natural "next pass" for the chapter and are scoped concretely in Section 7.6.